# 03 — Partie 2 : Fine-Tuning d'AraGPT2 (`aubmindlab/aragpt2-base`) & Génération de Paragraphes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinekamouss/fstt-dl-lab3-nlp-pytorch/blob/main/notebooks/03_part2_gpt2_finetuning.ipynb)

**Université Abdelmalek Essaadi — FST Tanger — Master MBD — Deep Learning (Lab 3)**

Note technique : la bibliothèque historique `pytorch-transformers` mentionnée dans l'énoncé est devenue **Hugging Face `transformers`** (`GPT2LMHeadModel` + `AutoTokenizer`). Nous adaptons l'architecture du tutoriel de référence (`mf1024`) aux standards modernes sur GPU Colab T4 (FP16, `AdamW`, cosine warmup, gradient accumulation, sauvegarde du meilleur checkpoint) avec le modèle arabe natif **`aubmindlab/aragpt2-base`** (135M paramètres) fine-tuné sur notre corpus Wikipédia arabe collecté en Partie 1.

## 1. Bootstrap Google Colab / Drive & Environnement

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None
REPO_URL = "https://github.com/yassinekamouss/fstt-dl-lab3-nlp-pytorch.git"
REPO_NAME = "fstt-dl-lab3-nlp-pytorch"
USE_DRIVE = True

if IN_COLAB:
    repo_dir = Path("/content") / REPO_NAME
    if not repo_dir.exists():
        !git clone {REPO_URL} {repo_dir}
    else:
        !git -C {repo_dir} pull --ff-only
    os.chdir(repo_dir)
else:
    cwd = Path.cwd().resolve()
    repo_dir = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

In [ ]:
if IN_COLAB:
    !pip install -q transformers datasets accelerate arabert sacrebleu arabic-reshaper python-bidi pyyaml nltk

import json
import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.colab_utils import get_device, load_config, print_system_info, seed_everything, setup_environment

paths = setup_environment(
    use_drive=USE_DRIVE,
    drive_dir="/content/drive/MyDrive/fstt-dl-lab3-nlp-pytorch",
    repo_root=Path.cwd(),
)
cfg = load_config(paths["config"])
seed_everything(cfg["project"]["seed"])
device = get_device()
print_system_info()

## 2. Chargement de `aubmindlab/aragpt2-base` & Préparation du Corpus Wikipédia Arabe

Contrairement au classifieur qui utilise des lemmes sans stop-words, le modèle génératif GPT-2 nécessite un **arabe naturel complet** (avec sa syntaxe et ses mots de liaison) afin de générer des paragraphes fluides et grammaticalement corrects.

In [ ]:
from src.gpt2_utils import build_lm_datasets, load_aragpt2

gpt2_cfg = cfg["part2_gpt2"]
model_name = gpt2_cfg["model_name"]

train_df = pd.read_csv(paths["data_processed"] / "train.csv")
val_df = pd.read_csv(paths["data_processed"] / "val.csv")
test_df = pd.read_csv(paths["data_processed"] / "test.csv")

# Privilegier les paragraphes pertinents (core + related) pour un fine-tuning specialise IA
train_texts = train_df[train_df["tier"].isin(["core", "related"])]["text"].astype(str).tolist()
val_texts = val_df[val_df["tier"].isin(["core", "related"])]["text"].astype(str).tolist()
test_texts = test_df[test_df["tier"].isin(["core", "related"])]["text"].astype(str).tolist()

model, tokenizer, arabert_prep = load_aragpt2(model_name=model_name)
model = model.to(device)

train_ds, val_ds, test_ds = build_lm_datasets(
    train_texts=train_texts,
    val_texts=val_texts,
    test_texts=test_texts,
    tokenizer=tokenizer,
    block_size=int(gpt2_cfg["block_size"]),
    arabert_prep=arabert_prep,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modele charge : {model_name} ({n_params:,} parametres)")
print(f"Blocs de {gpt2_cfg['block_size']} tokens -> Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)}")

## 3. Évaluation Baseline (Avant Fine-Tuning)

Nous mesurons la **Perplexity**, le **BLEU**, le **ROUGE (adapté arabe)** et le **Distinct-1/2** avant tout fine-tuning pour quantifier précisément l'apport de l'adaptation au domaine IA.

In [ ]:
from src.gpt2_utils import (
    build_trainer,
    build_training_arguments,
    evaluate_generation_quality,
    evaluate_perplexity,
    generate_paragraph,
)

gpt2_out_dir = paths["checkpoints"] / "aragpt2_runs"
training_args = build_training_arguments(output_dir=gpt2_out_dir, cfg=cfg, use_fp16=True)
trainer = build_trainer(
    model=model,
    args=training_args,
    train_ds=train_ds,
    val_ds=val_ds,
    tokenizer=tokenizer,
)

baseline_val_ppl = evaluate_perplexity(trainer, val_ds)
baseline_test_ppl = evaluate_perplexity(trainer, test_ds)
gen_cfg = gpt2_cfg["generation"]

baseline_nlg = evaluate_generation_quality(
    model=model,
    tokenizer=tokenizer,
    test_texts=test_texts,
    device=device,
    arabert_prep=arabert_prep,
    strategy="nucleus",
    n_samples=25,
    max_new_tokens=int(gen_cfg["max_new_tokens"]),
    min_new_tokens=int(gen_cfg["min_new_tokens"]),
    temperature=float(gen_cfg["temperature"]),
    top_k=int(gen_cfg["top_k"]),
    top_p=float(gen_cfg["top_p"]),
    repetition_penalty=float(gen_cfg["repetition_penalty"]),
    no_repeat_ngram_size=int(gen_cfg["no_repeat_ngram_size"]),
)

print(f"Baseline Val Perplexity  : {baseline_val_ppl:.2f}")
print(f"Baseline Test Perplexity : {baseline_test_ppl:.2f}")
print("Baseline NLG (Nucleus)   :", {k: round(v, 4) for k, v in baseline_nlg.items() if isinstance(v, (int, float))})

seed_prompts = list(gpt2_cfg["seed_prompts"])
baseline_generations = {
    prompt: generate_paragraph(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        strategy="nucleus",
        arabert_prep=arabert_prep,
        max_new_tokens=int(gen_cfg["max_new_tokens"]),
        min_new_tokens=int(gen_cfg["min_new_tokens"]),
        temperature=float(gen_cfg["temperature"]),
        top_k=int(gen_cfg["top_k"]),
        top_p=float(gen_cfg["top_p"]),
        repetition_penalty=float(gen_cfg["repetition_penalty"]),
        no_repeat_ngram_size=int(gen_cfg["no_repeat_ngram_size"]),
    )
    for prompt in seed_prompts
}

## 4. Fine-Tuning d'AraGPT2 sur le Corpus Wikipédia Arabe (FP16)

In [ ]:
train_result = trainer.train()
best_model_dir = paths["checkpoints"] / "aragpt2_finetuned"
trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))
print(f"Modele fine-tune sauvegarde dans : {best_model_dir}")

# Extraction et trace des courbes Train / Eval Loss
log_df = pd.DataFrame(trainer.state.log_history)
train_logs = log_df[log_df["loss"].notna()] if "loss" in log_df.columns else pd.DataFrame()
eval_logs = log_df[log_df["eval_loss"].notna()] if "eval_loss" in log_df.columns else pd.DataFrame()

fig, ax = plt.subplots(figsize=(9, 4.5))
if not train_logs.empty:
    ax.plot(train_logs["step"], train_logs["loss"], label="Train Loss", color="#2563eb", linewidth=2)
if not eval_logs.empty:
    ax.plot(eval_logs["step"], eval_logs["eval_loss"], label="Validation Loss", color="#10b981", marker="o", linewidth=2)
ax.set_title("Courbe d'Apprentissage — Fine-Tuning AraGPT2 (aubmindlab/aragpt2-base)")
ax.set_xlabel("Steps")
ax.set_ylabel("Cross-Entropy Loss")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()
fig.savefig(paths["figures"] / "part2_gpt2_loss.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Évaluation Après Fine-Tuning & Comparaison des Stratégies de Décodage

Comparaison quantitative **Avant vs Après Fine-Tuning**, ainsi que des 3 stratégies de génération : **Greedy**, **Beam Search**, et **Top-k / Top-p (Nucleus) Sampling**.

In [ ]:
finetuned_val_ppl = evaluate_perplexity(trainer, val_ds)
finetuned_test_ppl = evaluate_perplexity(trainer, test_ds)

strategy_results = {}
for strat in ("greedy", "beam", "nucleus"):
    strategy_results[strat] = evaluate_generation_quality(
        model=model,
        tokenizer=tokenizer,
        test_texts=test_texts,
        device=device,
        arabert_prep=arabert_prep,
        strategy=strat,
        n_samples=25,
        max_new_tokens=int(gen_cfg["max_new_tokens"]),
        min_new_tokens=int(gen_cfg["min_new_tokens"]),
        temperature=float(gen_cfg["temperature"]),
        top_k=int(gen_cfg["top_k"]),
        top_p=float(gen_cfg["top_p"]),
        repetition_penalty=float(gen_cfg["repetition_penalty"]),
        no_repeat_ngram_size=int(gen_cfg["no_repeat_ngram_size"]),
        num_beams=int(gen_cfg["num_beams"]),
    )

comparison_table = pd.DataFrame(
    [
        {
            "configuration": "Baseline (Pretrained) — Nucleus",
            "val_perplexity": baseline_val_ppl,
            "test_perplexity": baseline_test_ppl,
            "bleu_1": baseline_nlg["bleu_1"],
            "bleu_4": baseline_nlg["bleu_4"],
            "sacrebleu": baseline_nlg["sacrebleu"],
            "rouge1": baseline_nlg["rouge1"],
            "rougeL": baseline_nlg["rougeL"],
            "distinct_1": baseline_nlg["distinct_1"],
            "distinct_2": baseline_nlg["distinct_2"],
        },
        *[
            {
                "configuration": f"Fine-Tuned — {strat.title()}",
                "val_perplexity": finetuned_val_ppl,
                "test_perplexity": finetuned_test_ppl,
                "bleu_1": res["bleu_1"],
                "bleu_4": res["bleu_4"],
                "sacrebleu": res["sacrebleu"],
                "rouge1": res["rouge1"],
                "rougeL": res["rougeL"],
                "distinct_1": res["distinct_1"],
                "distinct_2": res["distinct_2"],
            }
            for strat, res in strategy_results.items()
        ],
    ]
)
display(comparison_table.round(4))

part2_report = {
    "model_name": model_name,
    "n_train_blocks": len(train_ds),
    "n_val_blocks": len(val_ds),
    "n_test_blocks": len(test_ds),
    "baseline": {
        "val_perplexity": baseline_val_ppl,
        "test_perplexity": baseline_test_ppl,
        "nlg": baseline_nlg,
    },
    "finetuned": {
        "val_perplexity": finetuned_val_ppl,
        "test_perplexity": finetuned_test_ppl,
        "strategies": strategy_results,
    },
}
(paths["reports"] / "metrics_part2.json").write_text(
    json.dumps(part2_report, ensure_ascii=False, indent=2), encoding="utf-8"
)

In [ ]:
# Comparaison qualitative sur les 5 phrases d'amorce arabes
showcase_rows = []
for prompt in seed_prompts:
    row = {
        "Prompt (Amorce)": prompt,
        "Baseline (Nucleus)": baseline_generations[prompt],
    }
    for strat in ("greedy", "beam", "nucleus"):
        row[f"Fine-Tuned ({strat.title()})"] = generate_paragraph(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            strategy=strat,
            arabert_prep=arabert_prep,
            max_new_tokens=int(gen_cfg["max_new_tokens"]),
            min_new_tokens=int(gen_cfg["min_new_tokens"]),
            temperature=float(gen_cfg["temperature"]),
            top_k=int(gen_cfg["top_k"]),
            top_p=float(gen_cfg["top_p"]),
            repetition_penalty=float(gen_cfg["repetition_penalty"]),
            no_repeat_ngram_size=int(gen_cfg["no_repeat_ngram_size"]),
            num_beams=int(gen_cfg["num_beams"]),
        )
    showcase_rows.append(row)

for idx, item in enumerate(showcase_rows, start=1):
    print("=" * 80)
    print(f"Exemple {idx} | Amorce : {item['Prompt (Amorce)']}")
    print(f"\n[Baseline Nucleus]   : {item['Baseline (Nucleus)']}")
    print(f"\n[Fine-Tuned Greedy]  : {item['Fine-Tuned (Greedy)']}")
    print(f"\n[Fine-Tuned Beam]    : {item['Fine-Tuned (Beam)']}")
    print(f"\n[Fine-Tuned Nucleus] : {item['Fine-Tuned (Nucleus)']}")

## 6. Démonstration Interactive & Synthèse Finale du Laboratoire

### Fonction de génération prête à l'emploi

In [ ]:
def demo_generate(seed_sentence: str, strategy: str = "nucleus") -> str:
    """Genere un nouveau paragraphe arabe coherent a partir d'une phrase d'amorce."""
    return generate_paragraph(
        model=model,
        tokenizer=tokenizer,
        prompt=seed_sentence,
        device=device,
        strategy=strategy,
        arabert_prep=arabert_prep,
        max_new_tokens=int(gen_cfg["max_new_tokens"]),
        min_new_tokens=int(gen_cfg["min_new_tokens"]),
        temperature=float(gen_cfg["temperature"]),
        top_k=int(gen_cfg["top_k"]),
        top_p=float(gen_cfg["top_p"]),
        repetition_penalty=float(gen_cfg["repetition_penalty"]),
        no_repeat_ngram_size=int(gen_cfg["no_repeat_ngram_size"]),
        num_beams=int(gen_cfg["num_beams"]),
    )

custom_prompt = "في مجال الرؤية الحاسوبية، تستطيع الشبكات العصبية الالتفافية"
print("Amorce  :", custom_prompt)
print("Genere  :", demo_generate(custom_prompt, strategy="nucleus"))

### Synthèse d'Apprentissage (Synthesis)

1. **Importance du préprocessing spécifique à l'arabe :** La richesse morphologique de l'arabe (*tashkeel*, variantes d'Alef, préfixes/suffixes collés) gonfle artificiellement la taille du vocabulaire. La normalisation (`PyArabic`) combinée à la lemmatisation (`Qalsadi`) réduit fortement la sparsité lexicale pour les modèles récurrents, tandis que le texte naturel non lemmatisé reste indispensable pour préserver la fluidité syntaxique lors du fine-tuning d'un LLM génératif (`AraGPT2`).
2. **Prévention de la fuite de données (*Data Leakage*) :** Regrouper le split `train/val/test` par titre d'article Wikipédia garantit une évaluation honnête sur des documents réellement inconnus, évitant le sur-apprentissage stylistique sur des paragraphes voisins.
3. **RNN vs Bi-RNN vs GRU vs LSTM :** Le `RNN` simple souffre de la dissipation du gradient sur des paragraphes longs ; le `Bi-RNN` exploite le contexte bidirectionnel en classification mais reste non-causal pour l'inférence autorégressive ; les mécanismes à portes (`GRU`, `LSTM`) capturent nettement mieux les dépendances longues tout en stabilisant l'entraînement avec `pack_padded_sequence` et le gradient clipping.
4. **Modèles séquentiels vs Transformers (`AraGPT2`) :** Le mécanisme d'auto-attention du Transformer et le pré-entraînement massif à sous-mots (BPE) permettent à `aubmindlab/aragpt2-base` d'atteindre une perplexité bien inférieure et de générer des paragraphes arabes cohérents après seulement quelques époques de fine-tuning, surtout avec le décodage **Nucleus (`top-p`) + `repetition_penalty`** qui équilibre fidélité terminologique et richesse lexicale (*Distinct-1/2*).